# GemmaFit - Cloud Training Pipeline v2

**Target:** fine-tune Gemma 4 E4B for local function-calling movement coaching.  
**Platform:** Colab Pro A100 preferred.  
**Current recipe:** P0 prompt-format expansion plus P1 domain-heavy mixture.

## v2 Changes

- Domain data file: `fc_training_data_chat.json`
- Domain rows are already chat-formatted as `messages`
- Domain/Glaive/HH mixture: `60/30/10`
- Run suffix: `gemmafit_v3_earlystop`
- Export names: `gemmafit-v3-earlystop-q4_k_m.gguf`, optional `gemmafit-v3-earlystop-q5_k_m.gguf`, and `gemmafit-v3-earlystop-fc.litertlm`

## Phases

| Section | Phase | Status |
| --- | --- | --- |
| 1 | Environment + secrets | run first |
| 1.5 | Smart HF model selection | run before model load |
| 2 | FC training data streams | v2 chat data |
| 3 | Video to landmark extraction | optional |
| 4 | Gemini synthetic expansion | skip for P0/P1 v2 |
| 5 | Load Gemma + Unsloth QLoRA | required |
| 5.5 | Pre-flight check | required before training |
| 6 | v2 training loop | run on Colab |
| 7 | Save adapter | after training |
| 8 | Convert to GGUF | llama.cpp fallback |
| 8.5 | Preserve HF merged export | required for LiteRT-LM conversion |
| 9 | Finalize LiteRT-LM artifact + smoke | required for Pixel AI path |

Keep v1 artifacts separate. Do not overwrite `gemmafit-q4_k_m.gguf`.


## Why this fork exists

This notebook is a non-destructive early-stop variant. It keeps the current
`train_gemma4_pipeline.ipynb` unchanged, uses separate checkpoint/adapter/GGUF
names, and is tuned to avoid the observed overfit pattern where train loss fell
near `0.036` while validation loss rose above `3.1`.

Main training changes:

- `max_steps = 1200` instead of 5000
- `eval_steps = save_steps = 100`
- `learning_rate = 5e-5`
- early stopping patience = 4 eval rounds
- save/load best checkpoint by `eval_loss`
- Q4 GGUF remains the primary export; Q5 is optional

## §1. Environment + Secrets

In [ ]:
%%capture
# Install — version pinning matters for Gemma 4 + Unsloth (Daniel Han / Unsloth tested combo).
try:
    import numpy, PIL
    _numpy = f"numpy=={numpy.__version__}"
    _pil   = f"pillow=={PIL.__version__}"
except Exception:
    _numpy, _pil = "numpy", "pillow"

!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
    unsloth "unsloth_zoo>=2026.4.6" "transformers==5.5.0" torchcodec timm \
    datasets huggingface_hub hf_transfer mediapipe opencv-python yt-dlp trl peft accelerate

import torch
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
# Expected: A100 40GB (Colab Pro) or 2× Tesla T4 (Kaggle Notebooks)


In [ ]:
# Colab secrets — set these in the side panel (🔑 icon) before running:
#   HF_TOKEN       — your HuggingFace token (read access; required for gated Gemma models)
#   GEMINI_API_KEY — for §4 only (defer until tomorrow)

from google.colab import userdata, drive
import os

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN') or ''
# os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')   # tomorrow

# Mount Drive first because this notebook is configured to persist BOTH:
#   1. Hugging Face downloaded base-model snapshots/cache
#   2. trained outputs: checkpoints, LoRA adapters, merged exports, logs
# Warning: base models can consume many GB of Drive storage.
drive.mount('/content/drive')

WORKDIR = '/content/drive/MyDrive/GemmaFit_train'
HF_CACHE_DIR = f'{WORKDIR}/hf_cache'
BASE_MODEL_DIR = f'{WORKDIR}/models/base_models'
TRAINED_OUTPUT_DIR = f'{WORKDIR}/trained_outputs'

for d in [
    WORKDIR,
    HF_CACHE_DIR,
    BASE_MODEL_DIR,
    TRAINED_OUTPUT_DIR,
    f'{WORKDIR}/landmarks',
    f'{TRAINED_OUTPUT_DIR}/checkpoints',
    f'{TRAINED_OUTPUT_DIR}/adapter',
    f'{TRAINED_OUTPUT_DIR}/merged_fp16',
    f'{TRAINED_OUTPUT_DIR}/logs',
]:
    os.makedirs(d, exist_ok=True)

# Faster / cleaner Hugging Face downloads.
# HF_HOME controls the Hugging Face cache location. Here it is intentionally placed on Drive.
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY', '1')

print('Workdir ready        :', WORKDIR)
print('HF cache on Drive    :', os.environ['HF_HOME'])
print('Base model directory :', BASE_MODEL_DIR)
print('Trained output dir   :', TRAINED_OUTPUT_DIR)


In [ ]:
# Pull the v2 chat-expanded FC examples from the GitHub repo so Colab has
# the same prompt formats used by Android production inference.

REPO_RAW = 'https://raw.githubusercontent.com/bkttt0429/GemmaFit/main'
FC_LOCAL = '/content/fc_training_data_chat.json'

!curl -sL {REPO_RAW}/finetune/data/fc_training_data_chat.json -o {FC_LOCAL}
!ls -lh {FC_LOCAL}

import json
fc = json.load(open(FC_LOCAL, encoding='utf-8'))
print('Local seeds:', fc['meta'])
print('Expansion:', fc['meta'].get('expansion', {}))
print('Train rows:', len(fc['train']))
print('Validation rows per format:', {k: len(v) for k, v in fc['validation'].items()})


## §1.5. Smart Hugging Face model selection + Drive auto-download

Run this after §1 install/secrets and before §5.

This cell does four things:

1. Detects GPU count, VRAM, RAM and free disk.
2. Chooses the safest Gemma model for the current runtime.
3. Checks Hugging Face token access before model loading.
4. Pre-downloads the selected model with `snapshot_download` into Google Drive, so §5 loads from a persisted local snapshot path.

The downstream cells use these globals:

| Global | Purpose |
| --- | --- |
| `RECOMMENDED_MODEL` | selected Hugging Face repo id |
| `LOCAL_MODEL_PATH` | Drive snapshot path if pre-download succeeds; otherwise repo id |
| `RECOMMENDED_SEQ_LEN` | max sequence length based on VRAM |
| `RECOMMENDED_BATCH` | per-device batch size based on VRAM |
| `GRAD_ACCUM` | gradient accumulation |
| `DEVICE_MAP` | `balanced` for multi-GPU, otherwise `None` |
| `TRAINED_OUTPUT_DIR` | Drive directory for checkpoints, adapters, merged exports and logs |
| `EST_STEPS_PER_SEC` | rough runtime estimate |

Default behavior in this notebook:

```python
SMART_DOWNLOAD_TO_DRIVE = True
```

So both the base model download and trained artifacts stay under:

```text
/content/drive/MyDrive/GemmaFit_train/
```


In [ ]:
# Smart Hugging Face model selection + Drive auto-download
import os, shutil, json, time, torch
from huggingface_hub import snapshot_download, HfApi
from huggingface_hub.utils import HfHubHTTPError, GatedRepoError, RepositoryNotFoundError

print('=== Smart HF model setup ===')

# ── User knobs ─────────────────────────────────────────────────────────────
PREFER_SMALL_MODEL = False          # True = faster dev: prefer E2B even on bigger GPUs
SMART_DOWNLOAD = True               # True = pre-download selected model before §5
SMART_DOWNLOAD_TO_DRIVE = True      # User setting: persist base model snapshot in Google Drive
HF_MODEL_ALLOW_PATTERNS = [
    '*.json', '*.model', '*.txt', '*.safetensors',
    'tokenizer*', 'generation_config.json', 'chat_template*',
]

# Optional manual override. Leave None for auto mode.
MANUAL_MODEL_NAME = None
# MANUAL_MODEL_NAME = 'unsloth/gemma-4-E2B-it'
# MANUAL_MODEL_NAME = 'unsloth/gemma-4-E4B-it'
# MANUAL_MODEL_NAME = 'google/gemma-4-E4B-it'  # canonical gated fp16; slower/heavier

HF_TOKEN = os.environ.get('HF_TOKEN') or None
WORKDIR = globals().get('WORKDIR', '/content/drive/MyDrive/GemmaFit_train')
BASE_MODEL_DIR = globals().get('BASE_MODEL_DIR', f'{WORKDIR}/models/base_models')
TRAINED_OUTPUT_DIR = globals().get('TRAINED_OUTPUT_DIR', f'{WORKDIR}/trained_outputs')
HF_CACHE_DIR = globals().get('HF_CACHE_DIR', f'{WORKDIR}/hf_cache')

for d in [WORKDIR, BASE_MODEL_DIR, TRAINED_OUTPUT_DIR, HF_CACHE_DIR]:
    os.makedirs(d, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_DIR

# ── Runtime detection ──────────────────────────────────────────────────────
def gib(x):
    return x / (1024 ** 3)

n_gpu = torch.cuda.device_count() if torch.cuda.is_available() else 0
gpu_names = [torch.cuda.get_device_name(i) for i in range(n_gpu)]
gpu_vram_gb = []
for i in range(n_gpu):
    props = torch.cuda.get_device_properties(i)
    gpu_vram_gb.append(gib(props.total_memory))

free_content_gb = gib(shutil.disk_usage('/content').free)
free_drive_gb = gib(shutil.disk_usage('/content/drive').free) if os.path.exists('/content/drive') else 0

print(f'GPU count       : {n_gpu}')
print(f'GPU names       : {gpu_names or ["CPU only"]}')
print(f'VRAM per GPU    : {[round(v, 1) for v in gpu_vram_gb]} GB')
print(f'Free /content   : {free_content_gb:.1f} GB')
print(f'Free Drive      : {free_drive_gb:.1f} GB')

# ── Model choice policy ────────────────────────────────────────────────────
max_vram = max(gpu_vram_gb) if gpu_vram_gb else 0
if MANUAL_MODEL_NAME:
    selected = MANUAL_MODEL_NAME
elif PREFER_SMALL_MODEL or max_vram < 18:
    selected = 'unsloth/gemma-4-E2B-it'
elif max_vram >= 24:
    selected = 'unsloth/gemma-4-E4B-it'
else:
    selected = 'unsloth/gemma-4-E2B-it'

# Training config policy.
if max_vram >= 40:
    seq_len, batch, grad_accum, est_sps = 2048, 4, 4, 6.0
elif max_vram >= 24:
    seq_len, batch, grad_accum, est_sps = 2048, 2, 8, 3.0
elif max_vram >= 15:
    seq_len, batch, grad_accum, est_sps = 1536, 1, 16, 1.5
else:
    seq_len, batch, grad_accum, est_sps = 1024, 1, 16, 0.7

device_map = 'balanced' if n_gpu > 1 else None

print('\nSelected model  :', selected)
print('Seq len / batch :', seq_len, '/', batch)
print('Grad accum      :', grad_accum)
print('Device map      :', device_map)

# ── Token / repo access check ──────────────────────────────────────────────
api = HfApi(token=HF_TOKEN)
try:
    info = api.model_info(selected)
    gated = getattr(info, 'gated', False)
    print(f'HF repo check   : OK  (gated={gated})')
except (GatedRepoError, RepositoryNotFoundError, HfHubHTTPError) as e:
    print('HF repo check   : FAILED')
    print('Reason          :', type(e).__name__, str(e)[:300])
    raise RuntimeError(
        'Cannot access selected model. Check HF_TOKEN, accept the model license on Hugging Face if gated, '
        'or set MANUAL_MODEL_NAME to an accessible repo.'
    )

# ── Smart pre-download ─────────────────────────────────────────────────────
local_model_path = selected
if SMART_DOWNLOAD:
    root = BASE_MODEL_DIR if SMART_DOWNLOAD_TO_DRIVE else '/content/hf_models'
    local_dir = os.path.join(root, selected.replace('/', '__'))
    os.makedirs(local_dir, exist_ok=True)

    if SMART_DOWNLOAD_TO_DRIVE and free_drive_gb < 20:
        print('\n⚠ Drive free space is below 20 GB. Base-model download may fail or fill Drive.')

    print('\nDownloading / syncing model snapshot...')
    print('Target          :', local_dir)
    print('Persist mode    :', 'Google Drive' if SMART_DOWNLOAD_TO_DRIVE else '/content temporary cache')
    print('HF cache        :', os.environ.get('HF_HOME'))
    t0 = time.time()
    try:
        local_model_path = snapshot_download(
            repo_id=selected,
            token=HF_TOKEN,
            local_dir=local_dir,
            local_dir_use_symlinks=False,
            allow_patterns=HF_MODEL_ALLOW_PATTERNS,
            resume_download=True,
            max_workers=8,
        )
        elapsed = time.time() - t0
        print(f'Download ready  : {local_model_path}')
        print(f'Elapsed         : {elapsed/60:.1f} min')
    except Exception as e:
        print('Download failed :', type(e).__name__, str(e)[:500])
        print('Fallback        : §5 will load directly from Hugging Face repo id.')
        local_model_path = selected

# ── Export globals used downstream ─────────────────────────────────────────
RECOMMENDED_MODEL = selected
LOCAL_MODEL_PATH = local_model_path
RECOMMENDED_SEQ_LEN = seq_len
RECOMMENDED_BATCH = batch
GRAD_ACCUM = grad_accum
DEVICE_MAP = device_map
EST_STEPS_PER_SEC = est_sps

SMART_HF_CONFIG = {
    'recommended_model': RECOMMENDED_MODEL,
    'local_model_path': LOCAL_MODEL_PATH,
    'seq_len': RECOMMENDED_SEQ_LEN,
    'batch': RECOMMENDED_BATCH,
    'grad_accum': GRAD_ACCUM,
    'device_map': DEVICE_MAP,
    'gpu_names': gpu_names,
    'vram_gb': gpu_vram_gb,
    'download_to_drive': SMART_DOWNLOAD_TO_DRIVE,
    'base_model_dir': BASE_MODEL_DIR,
    'hf_cache_dir': HF_CACHE_DIR,
    'trained_output_dir': TRAINED_OUTPUT_DIR,
}
print('\n=== Exported config ===')
print(json.dumps(SMART_HF_CONFIG, indent=2, ensure_ascii=False))


## §2. Phase A — FC training data via HuggingFace streaming

Three sources, all streamed (zero disk):

| Source | Role | Mix weight |
| --- | --- | --- |
| Your 600 FC seeds | domain-specific GemmaFit behaviour | 30% |
| `glaiveai/glaive-function-calling-v2` | general FC schema robustness | 60% |
| `Anthropic/hh-rlhf` filtered | refusal style for out-of-scope queries | 10% |

Streaming means: nothing downloads. Each batch is fetched on demand during training. Disk usage stays at ~5 MB regardless of dataset size.

In [ ]:
from datasets import Dataset, load_dataset, concatenate_datasets

# 2A - GemmaFit v2 domain seeds are already chat-formatted by format_expand.py.
# Rows are {messages, format, src_idx}; no fmt_domain mapping is needed.
domain_ds = load_dataset('json', data_files=FC_LOCAL, field='train', split='train')
print('Domain v2 chat data (eager):', len(domain_ds), 'examples')
print('Domain formats:', sorted(set(domain_ds['format'])))

# 2B - Glaive FC v2 (40K, stream)
glaive = load_dataset('glaiveai/glaive-function-calling-v2',
                      streaming=True, split='train')
print('Glaive (streaming): connected')

# 2C - HH-RLHF subset for refusal style (stream)
hh = load_dataset('Anthropic/hh-rlhf', streaming=True, split='train')
print('HH-RLHF (streaming): connected')


In [ ]:
# Inspect one sample from each source
import itertools

print('=== Domain seed ===')
print(json.dumps(domain_ds[0], ensure_ascii=False, indent=2)[:600])
print()
print('=== Glaive ===')
print(json.dumps(next(iter(glaive)), ensure_ascii=False, indent=2)[:600])
print()
print('=== HH-RLHF ===')
print(json.dumps(next(iter(hh)), ensure_ascii=False, indent=2)[:600])

In [ ]:
# Format external sources into the same chat schema.
# GemmaFit domain rows already contain a `messages` list from format_expand.py.
# Use a finite eager Dataset for training because current Unsloth SFTTrainer
# does not accept HuggingFace streaming mixer objects without batch_size.

def fmt_glaive(ex):
    """Glaive uses 'system' + 'chat' fields. Take the user/assistant turn."""
    sys_p = ex.get('system', '')
    chat = ex.get('chat', '')
    return {'messages': [
        {'role': 'system',    'content': sys_p[:500]},
        {'role': 'user',      'content': chat[:500]},
        {'role': 'assistant', 'content': chat[500:1500]},
    ]}


def fmt_hh(ex):
    """HH-RLHF has 'chosen' (preferred) and 'rejected' versions."""
    chosen = ex.get('chosen', '')
    parts = chosen.split('\n\nAssistant: ', 1)
    user_part = parts[0].replace('Human: ', '').strip()[:500]
    asst_part = (parts[1] if len(parts) > 1 else '').strip()[:500]
    return {'messages': [
        {'role': 'user',      'content': user_part},
        {'role': 'assistant', 'content': asst_part},
    ]}


domain_fmt = domain_ds
DOMAIN_ROWS = len(domain_fmt)
GLAIVE_ROWS = int(DOMAIN_ROWS * 0.30 / 0.60)  # 1020 when domain is 2040
HH_ROWS = int(DOMAIN_ROWS * 0.10 / 0.60)      # 340 when domain is 2040

print('Materializing external samples for finite SFT dataset...')
glaive_fmt = glaive.map(fmt_glaive, remove_columns=['system', 'chat'])
hh_fmt = hh.map(fmt_hh, remove_columns=['chosen', 'rejected'])

glaive_sample = Dataset.from_list(list(glaive_fmt.take(GLAIVE_ROWS)))
hh_sample = Dataset.from_list(list(hh_fmt.take(HH_ROWS)))

mixed = concatenate_datasets([domain_fmt, glaive_sample, hh_sample]).shuffle(seed=42)
print('Mixed finite dataset ready:')
print('  domain:', len(domain_fmt))
print('  glaive:', len(glaive_sample))
print('  hh    :', len(hh_sample))
print('  total :', len(mixed))
print('  ratio : 60/30/10 target')


In [ ]:
# Sanity-check: show 5 samples from the finite mixed dataset.
for i, ex in enumerate(mixed.select(range(min(5, len(mixed))))):
    print(f'[{i}]', json.dumps(ex, ensure_ascii=False)[:200])


In [ ]:
# Build eval splits from the v2 validation dictionary.
# Trainer uses production-format validation. Post-hoc eval_compare.py can test
# production/training-style prompts separately after GGUF export.

val_by_format = fc['validation']
assert set(val_by_format) >= {'production', 'bare', 'terse', 'chinese'}

EVAL_FORMAT = 'production'
eval_fmt = Dataset.from_list(val_by_format[EVAL_FORMAT])
print(f'Eval format: {EVAL_FORMAT}')
print(f'Eval (chat-formatted): {len(eval_fmt)} examples')
print('Validation rows per format:', {k: len(v) for k, v in val_by_format.items()})


## §3. Phase B — Video → landmark extraction

Goal: get pose-landmark JSONL from public exercise videos **without storing the videos**.

Strategy:

```
URL list  →  yt-dlp tmp.mp4  →  MediaPipe pose  →  landmarks.jsonl  →  rm tmp.mp4
```

Each clip occupies disk for ~30 seconds, then is deleted. Final output is
small JSONL files (~10 KB per clip) on Drive. These feed the
**Phase 3 Safety & Trust benchmark** (verify gates fire correctly on real bodies).

In [ ]:
# A starter list of public exercise videos. Add more as needed.
# Format: list of {url, exercise, view, source}

VIDEO_TARGETS = [
    # Squat — frontal
    {'url': 'https://commons.wikimedia.org/wiki/File:Squat-1.webm',
     'exercise': 'squat', 'view': 'frontal', 'source': 'wikimedia'},
    # Push-up
    {'url': 'https://commons.wikimedia.org/wiki/File:Push-ups.webm',
     'exercise': 'push_up', 'view': 'side', 'source': 'wikimedia'},
    # Add YouTube urls here, e.g.:
    # {'url': 'https://youtu.be/<id>', 'exercise': 'lunge', 'view': 'side', 'source': 'youtube'},
]
print(f'{len(VIDEO_TARGETS)} videos queued')

In [ ]:
import os, json, time, subprocess, hashlib
import cv2, mediapipe as mp

TMP_VIDEO = '/content/tmp_clip.mp4'
OUT_DIR = f'{WORKDIR}/landmarks'

LM_NAMES = [
    'nose','left_eye_inner','left_eye','left_eye_outer','right_eye_inner',
    'right_eye','right_eye_outer','left_ear','right_ear','mouth_left','mouth_right',
    'left_shoulder','right_shoulder','left_elbow','right_elbow',
    'left_wrist','right_wrist','left_pinky','right_pinky','left_index','right_index',
    'left_thumb','right_thumb','left_hip','right_hip','left_knee','right_knee',
    'left_ankle','right_ankle','left_heel','right_heel','left_foot_index','right_foot_index',
]

def download_clip(url: str, out_path: str = TMP_VIDEO) -> bool:
    if os.path.exists(out_path): os.remove(out_path)
    cmd = ['yt-dlp', '-q', '-f', 'best[height<=720]', '-o', out_path, url]
    r = subprocess.run(cmd, capture_output=True)
    return r.returncode == 0 and os.path.exists(out_path)


def extract_landmarks(video_path: str, sample_every: int = 2) -> list:
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frames = []
    pose = mp.solutions.pose.Pose(min_detection_confidence=0.5,
                                  min_tracking_confidence=0.5)
    idx = 0
    while cap.isOpened():
        ok, img = cap.read()
        if not ok: break
        if idx % sample_every == 0:
            res = pose.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            if res.pose_landmarks:
                frame = {'frame': idx, 'landmarks': {}}
                for i, lm in enumerate(res.pose_landmarks.landmark):
                    if i < len(LM_NAMES):
                        n = LM_NAMES[i]
                        frame['landmarks'][f'{n}_x'] = lm.x
                        frame['landmarks'][f'{n}_y'] = lm.y
                        frame['landmarks'][f'{n}_z'] = lm.z
                        frame['landmarks'][f'{n}_vis'] = lm.visibility
                frames.append(frame)
        idx += 1
    cap.release()
    pose.close()
    return frames, fps


def process_one(target: dict) -> dict:
    digest = hashlib.md5(target['url'].encode()).hexdigest()[:10]
    out_path = f'{OUT_DIR}/{target["exercise"]}_{digest}.jsonl'
    if os.path.exists(out_path):
        return {'status': 'cached', 'path': out_path}

    if not download_clip(target['url']):
        return {'status': 'download_failed'}
    try:
        frames, fps = extract_landmarks(TMP_VIDEO)
    except Exception as e:
        return {'status': f'extract_failed: {e}'}
    finally:
        if os.path.exists(TMP_VIDEO): os.remove(TMP_VIDEO)

    with open(out_path, 'w') as f:
        f.write(json.dumps({'meta': {**target, 'fps': fps,
                                      'frame_count': len(frames)}}) + '\n')
        for fr in frames:
            f.write(json.dumps(fr) + '\n')
    return {'status': 'ok', 'path': out_path, 'frames': len(frames)}


for tgt in VIDEO_TARGETS:
    t0 = time.time()
    res = process_one(tgt)
    dt = time.time() - t0
    print(f'[{dt:5.1f}s] {tgt["exercise"]:8s} {tgt["url"][:50]:50s} → {res}')

In [ ]:
# Inspect one extracted file
import os
files = [f for f in os.listdir(OUT_DIR) if f.endswith('.jsonl')]
print(f'{len(files)} landmark files:')
for f in files: print(f'  {f}: {os.path.getsize(os.path.join(OUT_DIR, f))/1024:.1f} KB')

if files:
    with open(os.path.join(OUT_DIR, files[0])) as fh:
        meta = json.loads(fh.readline())
        print()
        print('First file meta:', meta)

## §4. Phase C — Gemini synthetic expansion (TOMORROW)

Skip today. Plan:

1. Take each of your 600 FC seeds.
2. For each, ask Gemini Flash to rephrase the coaching message in 5 styles
   (formal / casual / encouraging / firm / multilingual zh-TW).
3. Write `/content/expanded_3000.jsonl` directly (no intermediate dump).
4. Cost estimate: 600 × 5 × ~500 tokens ≈ \$1-2 of Gemini API.

Why defer: the 600 + Glaive 40K is already enough to start a baseline run.
Spending API credits before you know whether the baseline works is premature.

In [ ]:
# DRY-RUN STUB — uncomment when ready (see §4 reasoning above).
#
# import google.generativeai as genai
# genai.configure(api_key=os.environ['GEMINI_API_KEY'])
# model = genai.GenerativeModel('gemini-2.0-flash')
# ...
print('§4 deferred to tomorrow.')

## §5. Phase D — Load Gemma 4 + Unsloth QLoRA

Today: load model, dry-run a forward pass to confirm GPU memory budget.
Don't train yet — wait for §4 expanded data so we don't waste a training run.

Memory check on A100 40GB:
- Gemma 4 E2B in 4-bit ≈ 1.5 GB
- E4B in 4-bit ≈ 3 GB
- LoRA adapters: < 0.5 GB
- Activations + KV cache: 5-15 GB depending on seq_len
- Both fit comfortably on A100 40GB.

In [ ]:
from unsloth import FastModel
import os, torch

# Use §1.5 smart Hugging Face setup if available; otherwise fall back.
MODEL_NAME = globals().get('RECOMMENDED_MODEL', 'unsloth/gemma-4-E4B-it')
MODEL_SOURCE = globals().get('LOCAL_MODEL_PATH', MODEL_NAME)   # local snapshot path when pre-downloaded
MAX_SEQ_LEN = globals().get('RECOMMENDED_SEQ_LEN', 2048)
DEVICE_MAP = globals().get('DEVICE_MAP', 'balanced' if torch.cuda.device_count() > 1 else None)

# Override here if you want a specific model regardless of detection:
# MODEL_NAME = 'unsloth/gemma-4-E4B-it'
# MODEL_SOURCE = MODEL_NAME
# MODEL_NAME = 'unsloth/gemma-4-E2B-it'
# MODEL_SOURCE = MODEL_NAME

print(f'Loading model source: {MODEL_SOURCE}')
print(f'Model label         : {MODEL_NAME}')
print(f'seq_len={MAX_SEQ_LEN}, device_map={DEVICE_MAP}')

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_SOURCE,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    full_finetuning=False,
    device_map=DEVICE_MAP,
    token=os.environ.get('HF_TOKEN') or None,
)
print(f'Loaded label        : {MODEL_NAME}')
print(f'Loaded source       : {MODEL_SOURCE}')
print(f'GPU mem after load  : {torch.cuda.memory_allocated()/1e9:.2f} GB')
print(f'GPU count           : {torch.cuda.device_count()} (device_map={DEVICE_MAP})')


In [ ]:
# Add LoRA adapters via FastModel
model = FastModel.get_peft_model(
    model,
    r=16,                # LoRA rank — 16 is a good default for E2B/E4B
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0.0,    # 0 is fastest; raise to 0.05 if overfitting
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
    max_seq_length=MAX_SEQ_LEN,
)
trainable, total = 0, 0
for p in model.parameters():
    total += p.numel()
    if p.requires_grad: trainable += p.numel()
print(f'Trainable: {trainable/1e6:.2f}M / {total/1e9:.2f}B '
      f'({100*trainable/total:.2f}%)')


In [ ]:
# Quick generation sanity check on the BASE model (before any training).
# Supports both v2 chat-expanded rows ({messages, format, src_idx}) and the old
# v1 raw rows ({input, output}).
FastModel.for_inference(model)

seed = domain_ds[0]

if 'messages' in seed:
    messages = list(seed['messages'])
    expected_msg = messages[-1] if messages and messages[-1].get('role') == 'assistant' else None
    prompt_messages = messages[:-1] if expected_msg else messages
    expected_text = expected_msg['content'] if expected_msg else ''
    print('Seed format:', seed.get('format', 'unknown'))
else:
    user_msg = f"Motion report:\n{json.dumps(seed['input'], ensure_ascii=False)}"
    prompt_messages = [{'role': 'user', 'content': user_msg}]
    expected_text = json.dumps(seed['output'], ensure_ascii=False)

prompt = tokenizer.apply_chat_template(
    prompt_messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(text=prompt, return_tensors='pt').to('cuda')
out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
print('=== Base-model output (no fine-tune) ===')
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                       skip_special_tokens=True))
print()
print('=== Expected assistant output ===')
try:
    print(json.dumps(json.loads(expected_text), ensure_ascii=False, indent=2))
except Exception:
    print(expected_text)


## Section 5.5. Pre-flight check (run before Section 6)

Catch bad configs before training. This cell verifies:

1. Model loaded and on GPU.
2. Mixed finite dataset produces valid samples.
3. Tokenizer chat template renders correctly.
4. Drive is mounted and writable.
5. Eval set is loaded.

If any check fails, fix the issue before running Section 6.


In [ ]:
import os, json, torch

print('=== Pre-flight check ===\n')

# 1. Model + GPU
assert 'model' in dir(), 'Model not loaded - re-run Section 5'
gpu_mem = torch.cuda.memory_allocated() / 1e9
assert gpu_mem > 0.5, f'GPU memory unexpectedly low: {gpu_mem:.2f} GB'
print(f'Model on GPU ({gpu_mem:.2f} GB allocated)')

# 2. Mixed finite dataset produces samples
sample = mixed[0]
assert 'messages' in sample, 'Mixed dataset missing messages key'
assert len(sample['messages']) >= 2, 'Sample has fewer than 2 messages'
assert len(mixed) >= 3000, f'Mixed dataset too small: {len(mixed)}'
print(f'Mixed dataset ready: {len(mixed)} rows ({len(sample["messages"])} msgs in first)')

# 3. Tokenizer chat template
formatted = tokenizer.apply_chat_template(
    sample['messages'], tokenize=False, add_generation_prompt=False)
assert len(formatted) > 50, f'Empty chat template output: {formatted!r}'
print(f'Chat template renders ({len(formatted)} chars)')

# 4. Drive is mounted + writable
test_file = f'{WORKDIR}/_preflight_test.txt'
with open(test_file, 'w') as f:
    f.write('ok')
os.remove(test_file)
print(f'Drive writable: {WORKDIR}')

# 5. Eval set
assert 'eval_fmt' in dir(), 'eval_fmt not built - re-run the eval-split cell in Section 2'
assert len(eval_fmt) >= 10, f'Eval set too small: {len(eval_fmt)}'
print(f'Eval set ready ({len(eval_fmt)} examples)')

# 6. Estimate runtime
print('\n=== Runtime estimate ===')
n_gpu = torch.cuda.device_count()
gpu_name = torch.cuda.get_device_name(0)
if 'A100' in gpu_name:
    sps = 6.0
elif 'T4' in gpu_name:
    sps = 1.5
else:
    sps = 3.0
max_steps_for_estimate = globals().get('MAX_STEPS', 1200)
overnight_h = max_steps_for_estimate / sps / 3600
print(f'  GPU: {gpu_name} x {n_gpu}')
print(f'  Estimated: {sps:.1f} steps/sec -> {max_steps_for_estimate} steps in ~{overnight_h:.1f} hr')

print('\nAll checks passed. Run Section 6 to start v3 early-stop training.')

## -6. Phase E - v2 training loop

This run uses the P0/P1 remediation:

- `fc_training_data_chat.json` generated by `format_expand.py`
- GemmaFit domain examples already in `messages` format
- domain/Glaive/HH mixture weighted `60/30/10`
- run name: `gemmafit_v2_format_expand`

Keep v1 artifacts separate; do not overwrite the v1 GGUF.


In [ ]:
# v3 early-stop training with disconnect resilience.
# This fork is for the observed overfit pattern: very low train loss with high eval loss.
# It writes separate checkpoints/adapters and leaves v2 artifacts untouched.

RUN_TRAINING = True   # Set False for dry-run cell validation.

if not RUN_TRAINING:
    raise SystemExit('Training disabled. Set RUN_TRAINING=True before starting Section 6.')

from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
import os, glob, json, time, re, math

# Pick batch/accum from -1.5 (or sensible defaults)
BATCH        = globals().get('RECOMMENDED_BATCH', 4)
GRAD_ACCUM_  = globals().get('GRAD_ACCUM', 4)
SEQ_LEN      = globals().get('RECOMMENDED_SEQ_LEN', MAX_SEQ_LEN)

# Early-stop defaults: keep this run short and select by validation loss.
MAX_STEPS = globals().get('MAX_STEPS', 1200)
EVAL_STEPS = globals().get('EVAL_STEPS', 100)
SAVE_STEPS = globals().get('SAVE_STEPS', EVAL_STEPS)
LEARNING_RATE = globals().get('LEARNING_RATE', 5e-5)
WARMUP_STEPS = globals().get('WARMUP_STEPS', 50)
EARLY_STOPPING_PATIENCE = globals().get('EARLY_STOPPING_PATIENCE', 4)
HIGH_EVAL_LOSS_WARNING = globals().get('HIGH_EVAL_LOSS_WARNING', 2.0)

TRAINED_OUTPUT_DIR = globals().get('TRAINED_OUTPUT_DIR', f'{WORKDIR}/trained_outputs')
MODEL_TAG = re.sub(r'[^A-Za-z0-9_.-]+', '_', globals().get('MODEL_NAME', 'gemma_model'))
RUN_SUFFIX = 'gemmafit_v3_earlystop'
RUN_NAME = f'{MODEL_TAG}_{RUN_SUFFIX}'

CKPT_DIR = f'{TRAINED_OUTPUT_DIR}/checkpoints/{RUN_NAME}'
ADAPTER_PATH = f'{TRAINED_OUTPUT_DIR}/adapter/{RUN_NAME}_best_adapter'
LOG_DIR = f'{TRAINED_OUTPUT_DIR}/logs/{RUN_NAME}'

os.environ['TENSORBOARD_LOGGING_DIR'] = LOG_DIR

for d in [CKPT_DIR, ADAPTER_PATH, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

existing = sorted(glob.glob(f'{CKPT_DIR}/checkpoint-*'),
                  key=lambda p: int(p.rsplit('-', 1)[1]))
RESUME = existing[-1] if existing else None
if RESUME:
    print(f'Resuming v3 early-stop run from {RESUME}')
else:
    print('Fresh v3 early-stop run.')

print(f'Run name: {RUN_NAME}')
print(f'Batch: {BATCH} x accum {GRAD_ACCUM_} = effective {BATCH * GRAD_ACCUM_}')
print(f'Seq len: {SEQ_LEN}')
print(f'Max steps: {MAX_STEPS}')
print(f'Eval/save every: {EVAL_STEPS}/{SAVE_STEPS} steps')
print(f'Learning rate: {LEARNING_RATE}')
print(f'Early stopping patience: {EARLY_STOPPING_PATIENCE} eval rounds')
print(f'Checkpoint dir: {CKPT_DIR}')
print(f'Best adapter : {ADAPTER_PATH}')
print(f'Log dir      : {LOG_DIR}')

FastModel.for_training(model)

training_args = SFTConfig(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    max_steps=MAX_STEPS,
    logging_steps=10,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=4,
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    per_device_eval_batch_size=BATCH,
    bf16=True,
    optim='adamw_8bit',
    seed=42,
    max_seq_length=SEQ_LEN,
    dataset_text_field=None,
    packing=False,
    report_to='tensorboard',
    resume_from_checkpoint=RESUME,
    ignore_data_skip=True,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
)

# Unsloth SFTTrainer expects a list of strings after applying chat template.
def fmt_for_sft(examples):
    if isinstance(examples['messages'][0], dict):
        batch_msgs = [examples['messages']]
    else:
        batch_msgs = examples['messages']

    return [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        for msgs in batch_msgs
    ]


# Guard against stale Colab state: old notebooks built `mixed` with streaming
# streaming mixer datasets, which current Unsloth cannot prepare because the
# internal RandomlyCyclingMultiSourcesExamplesIterable has no batch_size.
def ensure_finite_mixed_dataset():
    from datasets import Dataset, concatenate_datasets, load_dataset

    cls_name = type(globals().get('mixed', None)).__name__
    has_streaming_iterable = hasattr(globals().get('mixed', None), '_ex_iterable')
    if cls_name == 'Dataset' and not has_streaming_iterable:
        print(f'Using finite mixed dataset: {len(mixed)} rows')
        return mixed

    print(f'Rebuilding finite mixed dataset for Unsloth (old mixed type: {cls_name})')

    if 'domain_ds' not in globals():
        raise RuntimeError('domain_ds is missing. Re-run Section 2 data loading first.')
    if 'messages' not in domain_ds[0]:
        raise RuntimeError('domain_ds is not v2 chat data. Re-run Section 2 with fc_training_data_chat.json.')

    def _fmt_glaive(ex):
        sys_p = ex.get('system', '')
        chat = ex.get('chat', '')
        return {'messages': [
            {'role': 'system',    'content': sys_p[:500]},
            {'role': 'user',      'content': chat[:500]},
            {'role': 'assistant', 'content': chat[500:1500]},
        ]}

    def _fmt_hh(ex):
        chosen = ex.get('chosen', '')
        parts = chosen.split('\n\nAssistant: ', 1)
        user_part = parts[0].replace('Human: ', '').strip()[:500]
        asst_part = (parts[1] if len(parts) > 1 else '').strip()[:500]
        return {'messages': [
            {'role': 'user',      'content': user_part},
            {'role': 'assistant', 'content': asst_part},
        ]}

    domain_fmt_local = domain_ds
    domain_rows = len(domain_fmt_local)
    glaive_rows = int(domain_rows * 0.30 / 0.60)
    hh_rows = int(domain_rows * 0.10 / 0.60)

    glaive_stream = load_dataset('glaiveai/glaive-function-calling-v2', streaming=True, split='train') \
        .map(_fmt_glaive, remove_columns=['system', 'chat'])
    hh_stream = load_dataset('Anthropic/hh-rlhf', streaming=True, split='train') \
        .map(_fmt_hh, remove_columns=['chosen', 'rejected'])

    glaive_sample = Dataset.from_list(list(glaive_stream.take(glaive_rows)))
    hh_sample = Dataset.from_list(list(hh_stream.take(hh_rows)))
    rebuilt = concatenate_datasets([domain_fmt_local, glaive_sample, hh_sample]).shuffle(seed=42)

    print('Finite mixed dataset rebuilt:')
    print('  domain:', len(domain_fmt_local))
    print('  glaive:', len(glaive_sample))
    print('  hh    :', len(hh_sample))
    print('  total :', len(rebuilt))
    return rebuilt

mixed = ensure_finite_mixed_dataset()


trainer = SFTTrainer(
    model=model,
    train_dataset=mixed,
    eval_dataset=eval_fmt,
    args=training_args,
    formatting_func=fmt_for_sft,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        early_stopping_threshold=0.0,
    )],
)

t0 = time.time()
result = trainer.train(resume_from_checkpoint=RESUME)
elapsed = time.time() - t0

# load_best_model_at_end=True means this saves the best eval_loss checkpoint,
# not necessarily the final step. That is the intended anti-overfit behavior.
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
trainer.save_state()

eval_history = [h for h in trainer.state.log_history if 'eval_loss' in h]
best_eval = min(eval_history, key=lambda h: h['eval_loss']) if eval_history else {}
best_eval_loss = best_eval.get('eval_loss')
best_eval_step = best_eval.get('step')
perplexity = math.exp(best_eval_loss) if best_eval_loss is not None and best_eval_loss < 20 else None

done_marker = f'{TRAINED_OUTPUT_DIR}/TRAINING_DONE_{RUN_NAME}.json'
with open(done_marker, 'w') as f:
    json.dump({
        'version': 'v3_earlystop',
        'finished_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        'elapsed_seconds': int(elapsed),
        'final_train_loss': float(result.training_loss),
        'best_eval_loss': best_eval_loss,
        'best_eval_step': best_eval_step,
        'best_eval_perplexity': perplexity,
        'best_checkpoint': trainer.state.best_model_checkpoint,
        'global_step': result.global_step,
        'adapter_path': ADAPTER_PATH,
        'checkpoint_dir': CKPT_DIR,
        'log_dir': LOG_DIR,
        'effective_batch': BATCH * GRAD_ACCUM_,
        'seq_len': SEQ_LEN,
        'learning_rate': LEARNING_RATE,
        'max_steps': MAX_STEPS,
        'eval_steps': EVAL_STEPS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'model': MODEL_NAME,
        'model_source': globals().get('MODEL_SOURCE', MODEL_NAME),
        'trained_output_dir': TRAINED_OUTPUT_DIR,
        'domain_data': 'fc_training_data_chat.json',
        'mixture_probabilities': {'domain': 0.60, 'glaive': 0.30, 'hh_rlhf': 0.10},
        'eval_format': globals().get('EVAL_FORMAT', 'production'),
        'eval_history_tail': eval_history[-8:],
    }, f, indent=2)

print()
print('=' * 50)
print(f'  V3 EARLY-STOP TRAINING DONE in {elapsed/3600:.2f} hr')
print(f'  Final train loss: {result.training_loss:.4f}')
print(f'  Best eval loss:   {best_eval_loss} at step {best_eval_step}')
if perplexity is not None:
    print(f'  Best eval ppl:    {perplexity:.2f}')
print(f'  Best checkpoint:  {trainer.state.best_model_checkpoint}')
print(f'  Steps completed:  {result.global_step}')
print(f'  Adapter saved:    {ADAPTER_PATH}')
print(f'  Marker file:      {done_marker}')

if best_eval_loss is not None and best_eval_loss > HIGH_EVAL_LOSS_WARNING:
    print()
    print('WARNING: best eval_loss is still high. Do not ship only by loss; run the FC/safety validation cells before export.')

## §6.5. Reconnect-after-disconnect helper

Colab Pro sessions can drop after ~24 hr or on browser reload. Workflow:

1. **Reopen the notebook** → Runtime → Reconnect.
2. Re-run §1 (env + secrets + Drive mount).
3. Re-run §2 (data sources reconnect — streaming, no state to recover).
4. Re-run §5 (model loads fresh, ~2 min).
5. Re-run §5.5 (pre-flight check).
6. Re-run §6 with `RUN_TRAINING = True` → it auto-detects the latest
   checkpoint in Drive and resumes from there.

Skip §3 (landmarks) and §4 (synthetic) if those finished previously.

The cell below diagnoses your current state and detects whether
**training already finished** (via `TRAINING_DONE.json` marker).


In [ ]:
import os, glob, json, re

TRAINED_OUTPUT_DIR = globals().get('TRAINED_OUTPUT_DIR', f'{WORKDIR}/trained_outputs')
MODEL_TAG = re.sub(r'[^A-Za-z0-9_.-]+', '_', globals().get('MODEL_NAME', 'gemma_model'))
RUN_SUFFIX = 'gemmafit_v3_earlystop'
RUN_NAME = globals().get('RUN_NAME', f'{MODEL_TAG}_{RUN_SUFFIX}')

CKPT_DIR = f'{TRAINED_OUTPUT_DIR}/checkpoints/{RUN_NAME}'
DONE_MARKER = f'{TRAINED_OUTPUT_DIR}/TRAINING_DONE_{RUN_NAME}.json'

if os.path.exists(DONE_MARKER):
    with open(DONE_MARKER) as f:
        info = json.load(f)
    print('Training already finished!')
    for k, v in info.items():
        print(f'  {k}: {v}')
    print('\nSkip to -7 (adapter) or -8 (GGUF convert).')
else:
    ckpts = sorted(glob.glob(f'{CKPT_DIR}/checkpoint-*'),
                   key=lambda p: int(p.rsplit('-', 1)[1]))
    print('=== Resume diagnostic ===')
    print(f'Run name        : {RUN_NAME}')
    print(f'Drive workdir   : {WORKDIR}')
    print(f'Checkpoints dir : {CKPT_DIR}')
    print(f'Done marker     : {DONE_MARKER}')
    print(f'Found {len(ckpts)} checkpoint(s):')
    for c in ckpts[-5:]:
        step = c.rsplit('-', 1)[1]
        size_mb = sum(os.path.getsize(os.path.join(c, f))
                      for f in os.listdir(c)
                      if os.path.isfile(os.path.join(c, f))) / 1e6
        print(f'  step {step:>6}  ({size_mb:.1f} MB)  {c}')
    if ckpts:
        print(f'\nRe-run -1, -2, -5, -5.5, -6; it will resume from step {ckpts[-1].rsplit("-", 1)[1]}')
    else:
        print('  (none - fresh v3 early-stop run)')

## §7. Phase F — Save adapter, download to local

After training, only the LoRA adapter (~50 MB) gets persisted. Pull it back to your laptop manually from Drive.

In [ ]:
# Save LoRA adapter to the same Drive trained-output directory.
# If -6 already ran, ADAPTER_PATH points at the v2 final adapter.
TRAINED_OUTPUT_DIR = globals().get('TRAINED_OUTPUT_DIR', f'{WORKDIR}/trained_outputs')
os.makedirs(f'{TRAINED_OUTPUT_DIR}/adapter', exist_ok=True)

if 'ADAPTER_PATH' not in globals():
    RUN_NAME = globals().get('RUN_NAME', 'gemmafit_v3_earlystop')
    ADAPTER_PATH = f'{TRAINED_OUTPUT_DIR}/adapter/{RUN_NAME}_best_adapter'

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
!du -sh {ADAPTER_PATH}
print('Adapter saved to:', ADAPTER_PATH)

# Optional: also push to HuggingFace Hub so you can pull from anywhere.
# model.push_to_hub('<your-hf-username>/gemmafit-gemma4-fc-v2', token=os.environ['HF_TOKEN'])

## ?8. Phase G ? Convert to GGUF for llama.cpp deployment

Merge the best v3 early-stop LoRA into base and quantize `Q4_K_M` first for Android.
`Q5_K_M` remains optional and writes a separate v3 artifact name.


In [ ]:
# Merge best LoRA into base + save fp16 for GGUF / LiteRT-LM conversion.
# This v3 notebook writes separate merged exports and does not overwrite v2.
import glob, os, re, shutil

TRAINED_OUTPUT_DIR = globals().get('TRAINED_OUTPUT_DIR', f'{WORKDIR}/trained_outputs')
MERGED_ROOT = f'{TRAINED_OUTPUT_DIR}/merged_fp16'
os.makedirs(MERGED_ROOT, exist_ok=True)

MODEL_TAG = re.sub(r'[^A-Za-z0-9_.-]+', '_', globals().get('MODEL_NAME', 'gemma_model'))
RUN_SUFFIX = globals().get('RUN_SUFFIX', 'gemmafit_v3_earlystop')
RUN_NAME = globals().get('RUN_NAME', f'{MODEL_TAG}_{RUN_SUFFIX}')
MERGED_SLUG = re.sub(r'[^A-Za-z0-9_.-]+', '_', RUN_NAME)
MERGED_PATH = f'{MERGED_ROOT}/{MERGED_SLUG}_merged_fp16'
MERGED_TMP_PATH = f'/content/{MERGED_SLUG}_merged_fp16_tmp'
FORCE_REMERGE = globals().get('FORCE_REMERGE', False)

def safetensors_files(path):
    if not os.path.exists(path):
        return []
    return glob.glob(f'{path}/**/*.safetensors', recursive=True)

def zero_byte_safetensors(path):
    if not os.path.exists(path):
        return []
    return [
        p for p in safetensors_files(path)
        if os.path.isfile(p) and os.path.getsize(p) == 0
    ]

if os.path.exists(MERGED_PATH):
    stale_files = zero_byte_safetensors(MERGED_PATH)
    existing_files = safetensors_files(MERGED_PATH)
    if stale_files or not existing_files or FORCE_REMERGE:
        print('Removing stale merged export:', MERGED_PATH)
        if stale_files:
            print('Zero-byte safetensors found:', stale_files[:5])
        if not existing_files:
            print('No safetensors found in existing export.')
        shutil.rmtree(MERGED_PATH)
    else:
        print('Existing merged export found:', MERGED_PATH)

if FORCE_REMERGE or not os.path.exists(MERGED_PATH):
    shutil.rmtree(MERGED_TMP_PATH, ignore_errors=True)
    print('Merging to local runtime first:', MERGED_TMP_PATH)
    model.save_pretrained_merged(
        MERGED_TMP_PATH,
        tokenizer,
        save_method='merged_16bit',
        maximum_memory_usage=0.50,
    )
    stale_files = zero_byte_safetensors(MERGED_TMP_PATH)
    if stale_files:
        raise RuntimeError(f'Merged export contains zero-byte safetensors: {stale_files[:5]}')
    shutil.rmtree(MERGED_PATH, ignore_errors=True)
    shutil.copytree(MERGED_TMP_PATH, MERGED_PATH)
    shutil.rmtree(MERGED_TMP_PATH, ignore_errors=True)

stale_files = zero_byte_safetensors(MERGED_PATH)
if stale_files:
    raise RuntimeError(f'Drive merged export contains zero-byte safetensors: {stale_files[:5]}')

!du -sh {MERGED_PATH}
!ls -lh {MERGED_PATH}
print('Merged model saved to:', MERGED_PATH)

In [ ]:
# Convert merged to GGUF via llama.cpp.
# v3 names are explicit so v1/v2 GGUFs are not overwritten.
# Q4_K_M is the required Android artifact. Q5_K_M is optional because keeping
# fp16 + Q5 + Q4 at the same time can exceed Colab/Drive free space.
# Keep this cell defensive: llama.cpp requirements may upgrade transformers
# and expose a broken torchvision/torch pair in Colab, which prevents
# AutoTokenizer from importing Gemma4Config during vocab export.
import glob, os, shutil, subprocess, sys
from pathlib import Path

LLAMA_CPP_DIR = '/content/llama.cpp'
LOCAL_GGUF_DIR = '/content/gemmafit_gguf'
ARTIFACT_PREFIX = globals().get('ARTIFACT_PREFIX', 'gemmafit-v3-earlystop')
BUILD_Q5 = False  # Set True only when you also want the desktop Q5_K_M artifact.


def gib(num_bytes):
    return num_bytes / 1024**3


def print_disk(label, target):
    usage = shutil.disk_usage(target)
    print(f'{label}: free={gib(usage.free):.2f} GiB total={gib(usage.total):.2f} GiB path={target}')


def run_checked(cmd, *, cwd=None, label=None):
    print('\n$', ' '.join(map(str, cmd)))
    result = subprocess.run(
        cmd,
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout[-6000:])
    if result.returncode != 0:
        print_disk('/content disk', '/content')
        print_disk('WORKDIR disk', WORKDIR)
        raise RuntimeError(f'{label or cmd[0]} failed with exit code {result.returncode}')
    return result


def remove_if_exists(path):
    if os.path.exists(path):
        print(f'Removing stale file: {path}')
        os.remove(path)


def copy_to_drive(local_path, drive_path):
    if not os.path.exists(local_path) or os.path.getsize(local_path) == 0:
        raise FileNotFoundError(f'Missing local GGUF artifact: {local_path}')
    remove_if_exists(drive_path)
    shutil.copy2(local_path, drive_path)
    print(f'Copied {gib(os.path.getsize(drive_path)):.2f} GiB -> {drive_path}')


Path(LOCAL_GGUF_DIR).mkdir(parents=True, exist_ok=True)
print_disk('/content disk before GGUF export', '/content')
print_disk('WORKDIR disk before GGUF export', WORKDIR)

shutil.rmtree(LLAMA_CPP_DIR, ignore_errors=True)
run_checked([
    'git', 'clone', '--depth', '1',
    'https://github.com/ggerganov/llama.cpp', LLAMA_CPP_DIR,
], label='clone llama.cpp')

# Install converter dependencies, then restore the Unsloth-compatible
# transformers pin and remove optional vision packages not needed for GGUF.
run_checked([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    f'{LLAMA_CPP_DIR}/requirements.txt',
], label='install llama.cpp requirements')
run_checked([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==5.5.0', 'protobuf<5', 'numpy<2',
], label='restore notebook dependency pins')
subprocess.run([
    sys.executable, '-m', 'pip', 'uninstall', '-y', '-q',
    'torchvision', 'torchaudio',
], check=False)

os.environ['TRANSFORMERS_NO_TORCHVISION'] = '1'
os.environ['TRANSFORMERS_NO_TORCHAUDIO'] = '1'

GGUF_FP16 = f'{LOCAL_GGUF_DIR}/{ARTIFACT_PREFIX}-fp16.gguf'
GGUF_Q4_LOCAL = f'{LOCAL_GGUF_DIR}/{ARTIFACT_PREFIX}-q4_k_m.gguf'
GGUF_Q5_LOCAL = f'{LOCAL_GGUF_DIR}/{ARTIFACT_PREFIX}-q5_k_m.gguf'
GGUF_Q4 = f'{WORKDIR}/{ARTIFACT_PREFIX}-q4_k_m.gguf'
GGUF_Q5 = f'{WORKDIR}/{ARTIFACT_PREFIX}-q5_k_m.gguf'

for stale in [GGUF_FP16, GGUF_Q4_LOCAL, GGUF_Q5_LOCAL, GGUF_Q4]:
    remove_if_exists(stale)
if BUILD_Q5:
    remove_if_exists(GGUF_Q5)

convert_cmd = [
    sys.executable, f'{LLAMA_CPP_DIR}/convert_hf_to_gguf.py', MERGED_PATH,
    '--outfile', GGUF_FP16,
    '--outtype', 'f16',
]
run_checked(convert_cmd, label='convert_hf_to_gguf fp16')
if not os.path.exists(GGUF_FP16) or os.path.getsize(GGUF_FP16) == 0:
    raise FileNotFoundError(f'GGUF fp16 export was not created: {GGUF_FP16}')
print(f'FP16 local GGUF: {gib(os.path.getsize(GGUF_FP16)):.2f} GiB  {GGUF_FP16}')

run_checked(['cmake', '-B', 'build'], cwd=LLAMA_CPP_DIR, label='cmake configure llama.cpp')
run_checked([
    'cmake', '--build', 'build', '--target', 'llama-quantize', '-j',
], cwd=LLAMA_CPP_DIR, label='build llama-quantize')

quantize_bin = f'{LLAMA_CPP_DIR}/build/bin/llama-quantize'

# Build Q4 first: this is the artifact needed by Android/llama.cpp fallback.
run_checked([quantize_bin, GGUF_FP16, GGUF_Q4_LOCAL, 'Q4_K_M'], label='quantize Q4_K_M')
copy_to_drive(GGUF_Q4_LOCAL, GGUF_Q4)

if BUILD_Q5:
    run_checked([quantize_bin, GGUF_FP16, GGUF_Q5_LOCAL, 'Q5_K_M'], label='quantize Q5_K_M')
    copy_to_drive(GGUF_Q5_LOCAL, GGUF_Q5)
else:
    print('Skipping Q5_K_M. Set BUILD_Q5 = True in this cell if you need the optional desktop artifact.')

# Keep final Drive artifacts, then remove the large local fp16/temp files.
for temp_path in [GGUF_FP16, GGUF_Q4_LOCAL, GGUF_Q5_LOCAL]:
    remove_if_exists(temp_path)

print('\nFinal Drive GGUF artifacts:')
for path in sorted(glob.glob(f'{WORKDIR}/{ARTIFACT_PREFIX}-*.gguf')):
    print(f'{gib(os.path.getsize(path)):.2f} GiB  {path}')
print_disk('/content disk after cleanup', '/content')
print_disk('WORKDIR disk after GGUF export', WORKDIR)

## ?8.5. GemmaFit LiteRT-LM Android export

The Android app now prefers `models/gemmafit-v3-earlystop-fc.litertlm` through LiteRT-LM and keeps GGUF only as llama.cpp fallback. GGUF cannot be converted into LiteRT-LM; preserve the merged HF/safetensors export and convert that artifact with the official LiteRT-LM / AI Edge flow.

In [ ]:
# Preserve merged HF export metadata for LiteRT-LM conversion.
# The actual .litertlm conversion may run in this Colab or in the official AI Edge conversion notebook.
import glob, json, os, re, time

TRAINED_OUTPUT_DIR = globals().get('TRAINED_OUTPUT_DIR', f'{WORKDIR}/trained_outputs')
MODEL_TAG = re.sub(r'[^A-Za-z0-9_.-]+', '_', globals().get('MODEL_NAME', 'gemma_model'))
RUN_SUFFIX = globals().get('RUN_SUFFIX', 'gemmafit_v3_earlystop')
RUN_NAME = globals().get('RUN_NAME', f'{MODEL_TAG}_{RUN_SUFFIX}')
MERGED_SLUG = re.sub(r'[^A-Za-z0-9_.-]+', '_', RUN_NAME)
MERGED_PATH = globals().get('MERGED_PATH', f'{TRAINED_OUTPUT_DIR}/merged_fp16/{MERGED_SLUG}_merged_fp16')
ARTIFACT_PREFIX = globals().get('ARTIFACT_PREFIX', 'gemmafit-v3-earlystop')
LITERTLM_PATH = f'{WORKDIR}/{ARTIFACT_PREFIX}-fc.litertlm'
DONE_MARKER = f'{TRAINED_OUTPUT_DIR}/TRAINING_DONE_{RUN_NAME}.json'

if not os.path.isdir(MERGED_PATH):
    raise FileNotFoundError(f'Merged HF export not found: {MERGED_PATH}. Run Phase 8 first.')
zero_files = [
    p for p in glob.glob(f'{MERGED_PATH}/**/*.safetensors', recursive=True)
    if os.path.isfile(p) and os.path.getsize(p) == 0
]
if zero_files:
    raise RuntimeError(f'Merged HF export contains zero-byte safetensors: {zero_files[:5]}')

conversion_info = {
    'merged_hf_path': MERGED_PATH,
    'litertlm_path': LITERTLM_PATH,
    'conversion_status': 'not_started',
    'conversion_log': {
        'note': 'Convert from merged HF/safetensors export, not GGUF.',
        'target_artifact': f'models/{ARTIFACT_PREFIX}-fc.litertlm',
    },
    'tool_call_eval': None,
}
done = {}
if os.path.exists(DONE_MARKER):
    with open(DONE_MARKER) as f:
        done = json.load(f)
done.update(conversion_info)
done.setdefault('finished_at', time.strftime('%Y-%m-%d %H:%M:%S'))
with open(DONE_MARKER, "w") as f:
    json.dump(done, f, indent=2)

print('Merged HF export for LiteRT-LM:', MERGED_PATH)
print('Expected .litertlm output     :', LITERTLM_PATH)
print('Done marker updated          :', DONE_MARKER)

## ?9. LiteRT-LM smoke test and repo finalization

After conversion produces `gemmafit-v3-earlystop-fc.litertlm`, run the 8-tool smoke before copying the model to the Android app files directory.

In [ ]:
# Copy these helper scripts from the repo if this Colab runtime does not already have them.
# Local equivalent after downloading the artifact:
#   python finetune/prepare_litert_artifact.py --source-litertlm path/to/gemmafit-v3-earlystop-fc.litertlm --run-smoke

if os.path.exists(LITERTLM_PATH):
    print('LiteRT-LM artifact ready:', LITERTLM_PATH)
    print('Download it as models/gemmafit-v3-earlystop-fc.litertlm, then run the repo smoke script.')
else:
    print('Waiting for conversion output:', LITERTLM_PATH)

## §9. Reference — disk usage, troubleshooting, decisions

### Reference notebooks (Daniel Han / Unsloth)

| Notebook | Use |
| --- | --- |
| [Gemma 4 E4B — multimodal SFT](https://www.kaggle.com/code/danielhanchen/gemma4-e4b-unsloth) | Closest to our setup |
| [Gemma 4 31B — Kaggle 2× T4 fits](https://www.kaggle.com/code/danielhanchen/gemma4-31b-unsloth) | If you want the largest model |
| [Gemma 4 31B Vision SFT](https://www.kaggle.com/code/danielhanchen/gemma4-31b-vision-unsloth) | If pose-image multimodal needed |
| [All Unsloth notebooks](https://github.com/unslothai/notebooks/?tab=readme-ov-file#gemma-4-notebooks) | Audio / vision / text variants |

Optional tooling: [unsloth-buddy](https://github.com/TYH-labs/unsloth-buddy) — Claude Code skill that automates env setup, dataset reformatting, base-vs-FT comparison, and the side-by-side demo HTML the Kaggle judges expect.

### Disk usage budget

| Item | Size | Where |
| --- | --- | --- |
| Streaming buffers | < 100 MB | Colab `/tmp` (auto-cleared) |
| Domain seeds (your 600) | < 1 MB | `/content` |
| Landmark JSONL files | ~10 MB total | Drive `landmarks/` |
| Base 4-bit model | ~3 GB | GPU mem (not disk) |
| Checkpoints (3 kept) | ~150 MB | Drive `checkpoints/` |
| LoRA adapter (final) | ~50 MB | Drive `adapter/` |
| Merged fp16 (temp) | ~6 GB | Colab `/content` (delete after) |
| Final v3 Q4_K_M GGUF | ~1.5 GB | Drive |
| Optional v3 Q5_K_M GGUF | ~2 GB | Drive, only when `BUILD_Q5 = True` |
| **Persistent total in Drive** | **~1.5 GB Q4 only / ~3.7 GB with Q5** | |

Your laptop only ever pulls back the GGUFs (~3.5 GB) and the LoRA (~50 MB).

### Troubleshooting

| Symptom | Likely cause | Fix |
| --- | --- | --- |
| `OOM at training start` | seq_len or batch too big | drop `max_seq_length` to 1024, batch to 2 |
| `Streaming dataset hangs` | HF Hub rate limit | retry, or set `HF_TOKEN` |
| `Loss = NaN` | learning rate too high | LR 1e-4 instead of 2e-4 |
| `Trained model still bad JSON` | not enough domain weight | raise domain mix from 30% → 50% |
| `GGUF quantize fails` | architecture not yet supported | use Q8_0 as fallback |
| `FastLanguageModel not found` | older Unsloth | switch to `FastModel` (Gemma 4 path) |
| `transformers version mismatch` | unpinned install | re-run §1 install cell with pinned versions |

### Decision points (for tomorrow)

Before flipping `RUN_TRAINING = True`:

1. Did Phase A's `mixed` stream produce 5 sane samples in §2's preview?
2. Did Phase B's landmark extraction produce ≥ 3 JSONL files?
3. Did §5 base-model generation roughly resemble the expected output?
   - If output is nonsense → fine-tuning will help most.
   - If output is already 70%+ correct → consider skipping fine-tune, jump to LiteRT spike.

### Hardware: A100 vs Kaggle 2× T4

| Setup | Pros | Cons |
| --- | --- | --- |
| Colab Pro A100 40GB | Single GPU simple, big batch, fast | $10/mo, 24h sessions |
| Kaggle 2× T4 (free) | Free, 30hr/week, `device_map="balanced"` works | Slower, must split layers |

This notebook auto-detects: if `torch.cuda.device_count() > 1`, sets `device_map="balanced"` for Kaggle. Otherwise leaves it None for A100.

### Unsloth $10K Special Track eligibility

If you want to compete in the Unsloth Special Track ($10,000 prize):
- Use Unsloth for fine-tuning (this notebook does)
- Document Unsloth-specific optimisations (4-bit QLoRA, gradient checkpointing="unsloth")
- Submit base vs fine-tuned A/B comparison (use unsloth-buddy demo HTML, or write your own)